In [2]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer

from sklearn.ensemble import (
    GradientBoostingClassifier
)

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE



In [3]:

#CHARGEMENT DES DONNEES

path_cluster = "../data/processed/clusters_data.csv"

df = pd.read_csv(path_cluster)

print("****** Shape des données ******")
print(df.shape)

print("\n****** Colonnes ******")
print(df.columns)


****** Shape des données ******
(7043, 24)

****** Colonnes ******
Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn',
       'AvgMonthlySpend', 'Cluster', 'DBSCAN_Cluster'],
      dtype='object')


In [4]:


#PREPARATION DE X ET y

# Target
y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})


#X sans Cluster

X_without_cluster = df.drop(
    columns=[
        "Churn",
        "customerID",
        "Cluster",
        "DBSCAN_Cluster"
    ],
    errors="ignore"
)


#X avec Cluster

X_with_cluster = df.drop(
    columns=[
        "Churn",
        "customerID",
        "DBSCAN_Cluster"
    ],
    errors="ignore"
)


print("****** X sans Cluster ******")
print(X_without_cluster.columns)

print("****** X avec Cluster ******")
print(X_with_cluster.columns)



****** X sans Cluster ******
Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend'],
      dtype='object')
****** X avec Cluster ******
Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend', 'Cluster'],
      dtype='object')


In [5]:

#TRAIN / TEST SPLIT

X_train_without, X_test_without, y_train, y_test = train_test_split(
    X_without_cluster,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


X_train_with, X_test_with, _, _ = train_test_split(
    X_with_cluster,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)



In [6]:

#FONCTION DE PREPROCESSING

def create_preprocessor(X_train):

    numeric_features = X_train.select_dtypes(
        include=["int64", "float64"]
    ).columns

    categorical_features = X_train.select_dtypes(
        include=["object", "category"]
    ).columns


    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )


    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]
    )


    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numeric_pipeline,
                numeric_features
            ),
            (
                "cat",
                categorical_pipeline,
                categorical_features
            )
        ]
    )

    return preprocessor



In [7]:

#MODEL

model = GradientBoostingClassifier(
    random_state=42
)


#FONCTION POUR EVALUER UNE VERSION

def evaluate_version(
    X_train,
    X_test,
    y_train,
    y_test,
    version
):

    print("\n")
    print("=" * 60)
    print(version)
    print("=" * 60)


    preprocessor = create_preprocessor(
        X_train
    )


    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "smote",
                SMOTE(
                    random_state=42
                )
            ),
            (
                "model",
                model
            )
        ]
    )


    # Training
    pipeline.fit(
        X_train,
        y_train
    )


    # Prediction
    y_pred = pipeline.predict(
        X_test
    )


    # Probability Churn
    y_proba = pipeline.predict_proba(
        X_test
    )[:, 1]


    # Metrics
    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_proba
    )


    print("Precision :", round(precision, 4))
    print("Recall    :", round(recall, 4))
    print("F1-score  :", round(f1, 4))
    print("ROC-AUC   :", round(roc_auc, 4))


    return {
        "Version": version,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "ROC-AUC": roc_auc
    }



In [ ]:

#CLASSIFICATION SANS CLUSTER

result_without_cluster = evaluate_version(
    X_train_without,
    X_test_without,
    y_train,
    y_test,
    "Sans Cluster"
)


# CLASSIFICATION AVEC CLUSTER

result_with_cluster = evaluate_version(
    X_train_with,
    X_test_with,
    y_train,
    y_test,
    "Avec Cluster"
)





Sans Cluster
Precision : 0.5619
Recall    : 0.6791
F1-score  : 0.615
ROC-AUC   : 0.8412


Avec Cluster


In [ ]:

#COMPARAISON

results = pd.DataFrame([
    result_without_cluster,
    result_with_cluster
])


print("***********************COMPARAISON AVANT / APRES CLUSTER************")

print(results)


#CALCUL DELTA

without_cluster = (
    results[
        results["Version"] == "Sans Cluster"
    ]
    .iloc[0]
)

with_cluster = (
    results[
        results["Version"] == "Avec Cluster"
    ]
    .iloc[0]
)


comparison = pd.DataFrame({
    "Metric": [
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC"
    ],

    "Sans Cluster": [
        without_cluster["Precision"],
        without_cluster["Recall"],
        without_cluster["F1-score"],
        without_cluster["ROC-AUC"]
    ],

    "Avec Cluster": [
        with_cluster["Precision"],
        with_cluster["Recall"],
        with_cluster["F1-score"],
        with_cluster["ROC-AUC"]
    ]
})


comparison["Delta"] = (
    comparison["Avec Cluster"]
    -
    comparison["Sans Cluster"]
)


print("\n")
print("=" * 60)
print("             IMPACT DU CLUSTER")
print("=" * 60)

print(
    comparison.round(4)
)


#SAUVEGARDE

results.to_csv(
    "../data/processed/link_classification_results.csv",
    index=False
)

comparison.to_csv(
    "../data/processed/link_comparison.csv",
    index=False
)


print(
    "link_classification_results.csv créé avec succès !"
)

print(
    "link_comparison.csv créé avec succès !"
)